# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
%load_ext dotenv
%dotenv

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Load the PRICE_DATA 
PRICE_DATA = os.getenv("PRICE_DATA")

# Use glob to find all parquet files 
parquet_files = glob(os.path.join(PRICE_DATA, "**", "*.parquet"), recursive=True)



For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Read parquet files
dd_px = dd.read_parquet(parquet_files).set_index("ticker")

# Create a meta object based on the existing schema, with added columns - to address the warning
meta = dd_px._meta.assign(
    Close_lag_1 = "f8",
    Adj_Close_lag_1 = "f8",
    returns = "f8",
    hi_lo_range = "f8"
)

# Perform transformations 
dd_feat = dd_px.groupby("ticker", group_keys=False).apply(
    lambda x: x.assign(
        Close_lag_1 = x["Close"].shift(1),
        Adj_Close_lag_1 = x["Adj Close"].shift(1),
        returns = x["Close"] / x["Close"].shift(1) - 1,
        hi_lo_range = x["High"] - x["Low"]
    ),
    meta=meta
)


#AM Note: I specified metas after getting a warning: "`meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected."

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [5]:
# Convert Dask DataFrame to pandas
pdf_feat = dd_feat.compute()

# Sort by ticker and date (important for rolling operations)
pdf_feat = pdf_feat.sort_values(["ticker", "Date"])

# Add moving average of returns
pdf_feat["returns_ma10"] = pdf_feat.groupby("ticker")["returns"].transform(lambda x: x.rolling(10).mean())


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

In [6]:
# Check panda data frame properties
print(f"Rows: {pdf_feat.shape[0]}")
print(f"Columns: {pdf_feat.shape[1]}")
pdf_feat.info(memory_usage="deep")


Rows: 343915
Columns: 14
<class 'pandas.core.frame.DataFrame'>
Index: 343915 entries, A to ZEUS
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   Date             343915 non-null  datetime64[ns]
 1   Open             343880 non-null  float64       
 2   High             343880 non-null  float64       
 3   Low              343880 non-null  float64       
 4   Close            343880 non-null  float64       
 5   Adj Close        343880 non-null  float64       
 6   Volume           343880 non-null  float64       
 7   source           343915 non-null  string        
 8   Year             343915 non-null  int32         
 9   Close_lag_1      343790 non-null  float64       
 10  Adj_Close_lag_1  343790 non-null  float64       
 11  returns          343762 non-null  float64       
 12  hi_lo_range      343880 non-null  float64       
 13  returns_ma10     342371 non-null  float64       
dtypes:

No, it was not necessary to convert to pandas. Dask supports such operations. However, converting to pandas made the process simpler. Dask’s rolling operations are more complex to implement, especially with groupby, and often require index management and partition alignment. If the dataset fits in memory, using pandas is faster and more straightforward. Still, for very large datasets that do not fit in memory, it would have been better to use Dask to take advantage of its scalability and performance. Given that the final DataFrame is not too large (as seen above), using pandas was a reasonable and efficient choice in this case.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.